In [1]:
import numpy as np
import pandas as pd
import random

import torch
import torch.nn as nn
import torch.nn.functional as F

import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import date
import datetime

# Generating the training data for the Heat and Diffusion Model

In [2]:
data_dir = "./1D-AEMpy/"
depth_steps = 25 * 2 

print(os.getcwd())

D:\projects\1D-AEMpy\mcl\1_trainingData-MLP


In [3]:
meterological_data_df = pd.read_csv("./../output/lakes/mendota/output/py_meteorology_input.csv")
meterological_data_df = meterological_data_df # considering everything from 2nd time step

num_time_steps = meterological_data_df.shape[0]
depth_list = np.array(list(range(0, depth_steps)) * num_time_steps)*0.5+.25
depth_df = pd.DataFrame(data={'depth':depth_list})

#repeating the dataframe depth_steps number of times
meterological_data_df = pd.DataFrame(np.repeat(meterological_data_df.values, depth_steps, axis=0), columns=meterological_data_df.columns)
meterological_data_df = pd.concat([depth_df, meterological_data_df], ignore_index=False, axis=1)
meterological_data_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,icemovAvg,density_snow,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069


In [4]:
# Input INITIAL TEMP 00

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp_initial00.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_init00':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

,time,temp_init00,depth
0,2012-05-19 00:00:00,16.000000,0.25
1,2012-05-19 00:00:00,15.800000,0.75
2,2012-05-19 00:00:00,15.425000,1.25
3,2012-05-19 00:00:00,14.875000,1.75
4,2012-05-19 00:00:00,14.575000,2.25
...,...,...,...
2627995,2018-05-17 23:00:00,7.373164,22.75
2627996,2018-05-17 23:00:00,7.379868,23.25
2627997,2018-05-17 23:00:00,7.384346,23.75
2627998,2018-05-17 23:00:00,7.398099,24.25


In [5]:
final_df = meterological_data_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,density_snow,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,16.000000
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,15.800000
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,15.425000
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,14.875000
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,14.575000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.373164
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.379868
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.384346
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.398099


In [6]:
# Input HEAT TEMP 01

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp_heat01.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_heat01':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,ice_prior,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,16.000000,15.719326
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,15.800000,15.800009
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,15.425000,15.425005
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,14.875000,14.875005
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,14.575000,14.575005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.373164,7.373242
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.379868,7.379973
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.384346,7.384618
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.398099,7.398213


In [7]:
# Input ICE TEMP 02

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp_ice02.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_ice02':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,snow_prior,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,16.000000,15.719326,15.719326
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,15.800000,15.800009,15.800009
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,15.425000,15.425005,15.425005
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,14.875000,14.875005,14.875005
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.0,250.0,1.0,0.0,0.8,6.506215,14.575000,14.575005,14.575005
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.373164,7.373242,7.373242
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.379868,7.379973,7.379973
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.384346,7.384618,7.384618
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.0,250.0,1.0,0.0,0.8,15.707069,7.398099,7.398213,7.398213


In [8]:
# Input DIFF TEMP 03

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp_diff03.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_diff03':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,snowice_prior,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,250.0,1.0,0.0,0.8,6.506215,16.000000,15.719326,15.719326,15.719326
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,250.0,1.0,0.0,0.8,6.506215,15.800000,15.800009,15.800009,15.798172
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,250.0,1.0,0.0,0.8,6.506215,15.425000,15.425005,15.425005,15.424701
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,250.0,1.0,0.0,0.8,6.506215,14.875000,14.875005,14.875005,14.875998
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,250.0,1.0,0.0,0.8,6.506215,14.575000,14.575005,14.575005,14.575726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,250.0,1.0,0.0,0.8,15.707069,7.373164,7.373242,7.373242,7.373214
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,250.0,1.0,0.0,0.8,15.707069,7.379868,7.379973,7.379973,7.379952
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,250.0,1.0,0.0,0.8,15.707069,7.384346,7.384618,7.384618,7.384637
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,250.0,1.0,0.0,0.8,15.707069,7.398099,7.398213,7.398213,7.412002


In [9]:
# Input CONV TEMP 04

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp_conv04.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_conv04':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,rho_snow_prior,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,1.0,0.0,0.8,6.506215,16.000000,15.719326,15.719326,15.719326,15.757749
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,1.0,0.0,0.8,6.506215,15.800000,15.800009,15.800009,15.798172,15.757749
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,1.0,0.0,0.8,6.506215,15.425000,15.425005,15.425005,15.424701,15.424701
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,1.0,0.0,0.8,6.506215,14.875000,14.875005,14.875005,14.875998,14.875998
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,250.0,1.0,0.0,0.8,6.506215,14.575000,14.575005,14.575005,14.575726,14.575726
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,1.0,0.0,0.8,15.707069,7.373164,7.373242,7.373242,7.373214,7.373214
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,1.0,0.0,0.8,15.707069,7.379868,7.379973,7.379973,7.379952,7.379952
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,1.0,0.0,0.8,15.707069,7.384346,7.384618,7.384618,7.384637,7.388369
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,250.0,1.0,0.0,0.8,15.707069,7.398099,7.398213,7.398213,7.412002,7.388369


In [10]:
# Input MIX TEMP 05

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp_mix05.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'temp_mix05':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,IceSnowAttCoeff_prior,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,1.0,0.0,0.8,6.506215,16.000000,15.719326,15.719326,15.719326,15.757749,15.460315
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,1.0,0.0,0.8,6.506215,15.800000,15.800009,15.800009,15.798172,15.757749,15.460315
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,1.0,0.0,0.8,6.506215,15.425000,15.425005,15.425005,15.424701,15.424701,15.460315
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,1.0,0.0,0.8,6.506215,14.875000,14.875005,14.875005,14.875998,14.875998,15.460315
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,1.0,0.0,0.8,6.506215,14.575000,14.575005,14.575005,14.575726,14.575726,14.609642
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,1.0,0.0,0.8,15.707069,7.373164,7.373242,7.373242,7.373214,7.373214,7.373214
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,1.0,0.0,0.8,15.707069,7.379868,7.379973,7.379973,7.379952,7.379952,7.379952
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,1.0,0.0,0.8,15.707069,7.384346,7.384618,7.384618,7.384637,7.388369,7.388369
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,1.0,0.0,0.8,15.707069,7.398099,7.398213,7.398213,7.412002,7.388369,7.388369


In [11]:
# Input BUOYANCY

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_buoyancy.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'buoyancy':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,iceFlag_prior,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.8,6.506215,16.000000,15.719326,15.719326,15.719326,15.757749,15.460315,0.000000
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.8,6.506215,15.800000,15.800009,15.800009,15.798172,15.757749,15.460315,0.000000
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.8,6.506215,15.425000,15.425005,15.425005,15.424701,15.424701,15.460315,0.000000
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.8,6.506215,14.875000,14.875005,14.875005,14.875998,14.875998,15.460315,0.002525
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.0,0.8,6.506215,14.575000,14.575005,14.575005,14.575726,14.575726,14.609642,0.000242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.8,15.707069,7.373164,7.373242,7.373242,7.373214,7.373214,7.373214,0.000007
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.8,15.707069,7.379868,7.379973,7.379973,7.379952,7.379952,7.379952,0.000009
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.8,15.707069,7.384346,7.384618,7.384618,7.384637,7.388369,7.388369,0.000000
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.0,0.8,15.707069,7.398099,7.398213,7.398213,7.412002,7.388369,7.388369,0.001760


In [12]:
# Input DIFFUSIVITY

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_diff.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'diffusivity':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,dt_iceon_avg_prior,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.8,6.506215,16.000000,15.719326,15.719326,15.719326,15.757749,15.460315,0.000000,2.475864e-07
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.8,6.506215,15.800000,15.800009,15.800009,15.798172,15.757749,15.460315,0.000000,2.854859e-07
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.8,6.506215,15.425000,15.425005,15.425005,15.424701,15.424701,15.460315,0.000000,2.484661e-07
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.8,6.506215,14.875000,14.875005,14.875005,14.875998,14.875998,15.460315,0.002525,2.087484e-07
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.8,6.506215,14.575000,14.575005,14.575005,14.575726,14.575726,14.609642,0.000242,1.805459e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.8,15.707069,7.373164,7.373242,7.373242,7.373214,7.373214,7.373214,0.000007,2.800000e-07
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.8,15.707069,7.379868,7.379973,7.379973,7.379952,7.379952,7.379952,0.000009,2.800000e-07
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.8,15.707069,7.384346,7.384618,7.384618,7.384637,7.388369,7.388369,0.000000,2.800000e-07
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.8,15.707069,7.398099,7.398213,7.398213,7.412002,7.388369,7.388369,0.001760,2.800000e-07


In [13]:
# Input density gradient

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_density-conv.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'densityGradient':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,icemovAvg_prior,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,16.000000,15.719326,15.719326,15.719326,15.757749,15.460315,0.000000,2.475864e-07,0.012585
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,15.800000,15.800009,15.800009,15.798172,15.757749,15.460315,0.000000,2.854859e-07,-0.058974
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,15.425000,15.425005,15.425005,15.424701,15.424701,15.460315,0.000000,2.484661e-07,-0.083695
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,14.875000,14.875005,14.875005,14.875998,14.875998,15.460315,0.002525,2.087484e-07,-0.044301
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,6.506215,14.575000,14.575005,14.575005,14.575726,14.575726,14.609642,0.000242,1.805459e-07,-0.007368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,7.373164,7.373242,7.373242,7.373214,7.373214,7.373214,0.000007,2.800000e-07,0.000348
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,7.379868,7.379973,7.379973,7.379952,7.379952,7.379952,0.000009,2.800000e-07,0.000242
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,7.384346,7.384618,7.384618,7.384637,7.388369,7.388369,0.000000,2.800000e-07,0.001420
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,15.707069,7.398099,7.398213,7.398213,7.412002,7.388369,7.388369,0.001760,2.800000e-07,-0.090893


In [14]:
# Input temp change

out_temp_df = pd.read_csv("./../output/lakes/mendota/output/py_temp-conv.csv")

flattened_temp = out_temp_df.iloc[:,1:].to_numpy().flatten()

time_stamp = out_temp_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'tempChange':flattened_temp, 'depth':depth_list}
out_temp_df = pd.DataFrame(data=data)

out_temp_df

final_df = final_df.merge(out_temp_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_init00,temp_heat01,temp_ice02,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient,tempChange
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,16.000000,15.719326,15.719326,15.719326,15.757749,15.460315,0.000000,2.475864e-07,0.012585,15.798172
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,15.800000,15.800009,15.800009,15.798172,15.757749,15.460315,0.000000,2.854859e-07,-0.058974,15.424701
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,15.425000,15.425005,15.425005,15.424701,15.424701,15.460315,0.000000,2.484661e-07,-0.083695,14.875998
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,14.875000,14.875005,14.875005,14.875998,14.875998,15.460315,0.002525,2.087484e-07,-0.044301,14.575726
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,14.575000,14.575005,14.575005,14.575726,14.575726,14.609642,0.000242,1.805459e-07,-0.007368,14.525075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.373164,7.373242,7.373242,7.373214,7.373214,7.373214,0.000007,2.800000e-07,0.000348,7.379952
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.379868,7.379973,7.379973,7.379952,7.379952,7.379952,0.000009,2.800000e-07,0.000242,7.384637
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.384346,7.384618,7.384618,7.384637,7.388369,7.388369,0.000000,2.800000e-07,0.001420,7.412002
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.398099,7.398213,7.398213,7.412002,7.388369,7.388369,0.001760,2.800000e-07,-0.090893,3.923803


In [15]:
# ICE AND SNOW

ice_data_df = pd.read_csv("./../output/lakes/mendota/output/py_icesnow.csv")

#repeating the dataframe depth_steps number of times
ice_data_df = pd.DataFrame(np.repeat(ice_data_df.values, depth_steps, axis=0), columns=ice_data_df.columns)
ice_data_df = pd.concat([depth_df, ice_data_df], ignore_index=False, axis=1)
print(ice_data_df)

final_df = final_df.merge(ice_data_df, how='inner', on=['time','depth'])
final_df

         depth                 time  ice snow snowice
0         0.25  2012-05-19 00:00:00  0.0  0.0     0.0
1         0.75  2012-05-19 00:00:00  0.0  0.0     0.0
2         1.25  2012-05-19 00:00:00  0.0  0.0     0.0
3         1.75  2012-05-19 00:00:00  0.0  0.0     0.0
4         2.25  2012-05-19 00:00:00  0.0  0.0     0.0
...        ...                  ...  ...  ...     ...
2627995  22.75  2018-05-17 23:00:00  0.0  0.0     0.0
2627996  23.25  2018-05-17 23:00:00  0.0  0.0     0.0
2627997  23.75  2018-05-17 23:00:00  0.0  0.0     0.0
2627998  24.25  2018-05-17 23:00:00  0.0  0.0     0.0
2627999  24.75  2018-05-17 23:00:00  0.0  0.0     0.0

[2628000 rows x 5 columns]


,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,temp_diff03,temp_conv04,temp_mix05,buoyancy,diffusivity,densityGradient,tempChange,ice,snow,snowice
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,15.719326,15.757749,15.460315,0.000000,2.475864e-07,0.012585,15.798172,0.0,0.0,0.0
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,15.798172,15.757749,15.460315,0.000000,2.854859e-07,-0.058974,15.424701,0.0,0.0,0.0
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,15.424701,15.424701,15.460315,0.000000,2.484661e-07,-0.083695,14.875998,0.0,0.0,0.0
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,14.875998,14.875998,15.460315,0.002525,2.087484e-07,-0.044301,14.575726,0.0,0.0,0.0
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,14.575726,14.575726,14.609642,0.000242,1.805459e-07,-0.007368,14.525075,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.373214,7.373214,7.373214,0.000007,2.800000e-07,0.000348,7.379952,0.0,0.0,0.0
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.379952,7.379952,7.379952,0.000009,2.800000e-07,0.000242,7.384637,0.0,0.0,0.0
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.384637,7.388369,7.388369,0.000000,2.800000e-07,0.001420,7.412002,0.0,0.0,0.0
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,7.412002,7.388369,7.388369,0.001760,2.800000e-07,-0.090893,3.923803,0.0,0.0,0.0


In [16]:
# lake characteristics

ice_data_df = pd.read_csv("./../output/lakes/mendota/output/py_lakecharacteristics.csv")

#repeating the dataframe depth_steps number of times
ice_data_df = pd.DataFrame(np.repeat(ice_data_df.values, depth_steps, axis=0), columns=ice_data_df.columns)
ice_data_df = pd.concat([depth_df, ice_data_df], ignore_index=False, axis=1)
print(ice_data_df)

final_df = final_df.merge(ice_data_df, how='inner', on=['time','depth'])
final_df

         depth                 time     Volume_m2   Osgood MaxDepth_m  \
0         0.25  2012-05-19 00:00:00  486837500.75  2.93527      25.75   
1         0.75  2012-05-19 00:00:00  486837500.75  2.93527      25.75   
2         1.25  2012-05-19 00:00:00  486837500.75  2.93527      25.75   
3         1.75  2012-05-19 00:00:00  486837500.75  2.93527      25.75   
4         2.25  2012-05-19 00:00:00  486837500.75  2.93527      25.75   
...        ...                  ...           ...      ...        ...   
2627995  22.75  2018-05-17 23:00:00  486837500.75  2.93527      25.75   
2627996  23.25  2018-05-17 23:00:00  486837500.75  2.93527      25.75   
2627997  23.75  2018-05-17 23:00:00  486837500.75  2.93527      25.75   
2627998  24.25  2018-05-17 23:00:00  486837500.75  2.93527      25.75   
2627999  24.75  2018-05-17 23:00:00  486837500.75  2.93527      25.75   

        MeanDepth_m  
0          13.21675  
1          13.21675  
2          13.21675  
3          13.21675  
4          13

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,diffusivity,densityGradient,tempChange,ice,snow,snowice,Volume_m2,Osgood,MaxDepth_m,MeanDepth_m
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,2.475864e-07,0.012585,15.798172,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,2.854859e-07,-0.058974,15.424701,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,2.484661e-07,-0.083695,14.875998,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,2.087484e-07,-0.044301,14.575726,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,1.805459e-07,-0.007368,14.525075,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,2.800000e-07,0.000348,7.379952,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,2.800000e-07,0.000242,7.384637,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,2.800000e-07,0.001420,7.412002,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,2.800000e-07,-0.090893,3.923803,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675


In [17]:
temp_obs_df = pd.read_csv("./../output/lakes/mendota/output/py_observed_temp.csv")


flattened_temp = temp_obs_df.iloc[:,1:].to_numpy().flatten()
time_stamp = temp_obs_df['time'].repeat(depth_steps).values

data = {'time':time_stamp, 'obs_temp':flattened_temp, 'depth':depth_list}

temp_obs_df = pd.DataFrame(data=data)

temp_obs_df


final_df = final_df.merge(temp_obs_df, how='inner', on=['time','depth'])
final_df

,depth,time,AirTemp_degC,Longwave_Wm-2,Latent_Wm-2,Sensible_Wm-2,Shortwave_Wm-2,lightExtinct_m-1,TKE_Jm-1,ShearStress_Nm-2,...,densityGradient,tempChange,ice,snow,snowice,Volume_m2,Osgood,MaxDepth_m,MeanDepth_m,obs_temp
0,0.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,0.012585,15.798172,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,16.780000
1,0.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,-0.058974,15.424701,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,16.683000
2,1.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,-0.083695,14.875998,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,16.700000
3,1.75,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,-0.044301,14.575726,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,16.832000
4,2.25,2012-05-19 00:00:00,13.9303,-61.964873,-76.678734,-12.654331,0.0,0.6,21119384.885576,0.027871,...,-0.007368,14.525075,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,16.888000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2627995,22.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.000348,7.379952,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,7.092222
2627996,23.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.000242,7.384637,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,7.069167
2627997,23.75,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,0.001420,7.412002,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,7.069167
2627998,24.25,2018-05-17 23:00:00,13.02023,-59.698216,-60.503325,-20.211467,0.0,0.6,30278264.767321,0.035437,...,-0.090893,3.923803,0.0,0.0,0.0,486837500.75,2.93527,25.75,13.21675,7.069167


In [18]:
obs_array = final_df['obs_temp']
obs_array[obs_array == -999] = final_df['temp_mix05']
print(obs_array)
final_df['obs_temp'] = obs_array

0          16.780000
1          16.683000
2          16.700000
3          16.832000
4          16.888000
             ...    
2627995     7.092222
2627996     7.069167
2627997     7.069167
2627998     7.069167
2627999     7.074107
Name: obs_temp, Length: 2628000, dtype: float64


C:\Users\au740615\AppData\Local\Temp\ipykernel_35876\349148568.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obs_array[obs_array == -999] = final_df['temp_mix05']


In [19]:
final_df_null = final_df.fillna('')
print(final_df_null.head)

<bound method NDFrame.head of          depth                 time  AirTemp_degC  Longwave_Wm-2  Latent_Wm-2  \
0         0.25  2012-05-19 00:00:00      13.93030     -61.964873   -76.678734   
1         0.75  2012-05-19 00:00:00      13.93030     -61.964873   -76.678734   
2         1.25  2012-05-19 00:00:00      13.93030     -61.964873   -76.678734   
3         1.75  2012-05-19 00:00:00      13.93030     -61.964873   -76.678734   
4         2.25  2012-05-19 00:00:00      13.93030     -61.964873   -76.678734   
...        ...                  ...           ...            ...          ...   
2627995  22.75  2018-05-17 23:00:00      13.02023     -59.698216   -60.503325   
2627996  23.25  2018-05-17 23:00:00      13.02023     -59.698216   -60.503325   
2627997  23.75  2018-05-17 23:00:00      13.02023     -59.698216   -60.503325   
2627998  24.25  2018-05-17 23:00:00      13.02023     -59.698216   -60.503325   
2627999  24.75  2018-05-17 23:00:00      13.02023     -59.698216   -60.503325  

In [20]:
# iterating the columns
for col in final_df_null.columns:
    print(col)

depth
time
AirTemp_degC
Longwave_Wm-2
Latent_Wm-2
Sensible_Wm-2
Shortwave_Wm-2
lightExtinct_m-1
TKE_Jm-1
ShearStress_Nm-2
Area_m2
CC
ea
Jlw
Uw
Pa
RH
PP
IceSnowAttCoeff
iceFlag
icemovAvg
density_snow
ice_prior
snow_prior
snowice_prior
rho_snow_prior
IceSnowAttCoeff_prior
iceFlag_prior
dt_iceon_avg_prior
icemovAvg_prior
temp_init00
temp_heat01
temp_ice02
temp_diff03
temp_conv04
temp_mix05
buoyancy
diffusivity
densityGradient
tempChange
ice
snow
snowice
Volume_m2
Osgood
MaxDepth_m
MeanDepth_m
obs_temp


In [21]:
final_df_null.to_csv("mendota-all_data_lake_modeling_in_time.csv", index=False)